In [ ]:
# --- repo path bootstrap ---
import sys, pathlib
ROOT = next(p for p in pathlib.Path.cwd().parents if (p / "utils" / "paths.py").is_file())
sys.path.insert(0, str(ROOT))
from utils.paths import (profiles, features, feature_output, figdir, metadata,
                         data_dir, derived, require)
from utils.panels import save_panel

import pandas as pd
import numpy as np
import os

# Pycytominer
from pycytominer import feature_select
from pycytominer import normalize

# Set current working directory


In [ ]:
def list_features(df):
    # List features
    list_of_selected_features = list(df.columns.values)
    list_of_metadata = list(df.columns[df.columns.str.contains("Metadata_")])
    list_of_selected_features = list(set(list_of_selected_features) - set(list_of_metadata))
    
    return list_of_selected_features, list_of_metadata

In [ ]:
cell_line  = 'HCT116'
data_dir =str(features("exp1_main", "SingleSlice")) + "/"
n_slices  = 12
OutputDir = derived("exp1_main", "slices")   # derived/, never the deposit
if not os.path.exists(OutputDir):
    os.makedirs(OutputDir)


In [ ]:
files = [f for f in os.listdir(data_dir) if cell_line in f and 'MedianAgg' in f]
data = pd.concat([pd.read_parquet(data_dir + f) for f in files], ignore_index=True)
data['_col'] = data['Metadata_Well'].str[1:].astype(int)
data.rename(columns={'Metadata_Site': 'Metadata_z'}, inplace=True)
print(f'Loaded {data.shape[0]} wells x {data.shape[1]} columns')
print('Plates:', data['Metadata_Barcode'].unique().tolist())
print('Z-slices available:', sorted(data['Metadata_z'].unique().tolist()))

In [ ]:
def process_case(data, case_name, barcodes=None, col_range=None, z_slices=None):
    df = data.copy()
    if barcodes is not None:
        df = df[df['Metadata_Barcode'].isin(barcodes)]
    if col_range is not None:
        df = df[(df['_col'] >= col_range[0]) & (df['_col'] <= col_range[1])]
    if z_slices is not None:
        df = df[df['Metadata_z'].isin(z_slices)]
    if df.empty:
        print(f'[{case_name}] No data after filtering — skipping.')
        return None
    print(f'[{case_name}] {df["Metadata_Barcode"].nunique()} plate(s), '
          f'{df["Metadata_z"].nunique()} z-slice(s), {df["_col"].nunique()} well-cols')

    # Normalize per plate x z-slice
    df['Metadata_plate_slice'] = df['Metadata_Barcode'] + '_' + df['Metadata_z'].astype(str)
    features_list = list_features(df)[0]
    normalized_parts = []
    for unit in df['Metadata_plate_slice'].unique():
        temp = df[df['Metadata_plate_slice'] == unit]
        if temp[temp['Metadata_cmpdname'] == 'dmso'].empty:
            print(f'  [{case_name}] No DMSO in {unit} — skipping slice.')
            continue
        norm_temp = normalize(temp, features=features_list, image_features=False,
                              meta_features='infer',
                              samples="Metadata_cmpdname == 'dmso'",
                              method='standardize')
        normalized_parts.append(norm_temp)
    if not normalized_parts:
        print(f'[{case_name}] No slices could be normalized — skipping.')
        return None
    normalized = pd.concat(normalized_parts, ignore_index=True)

    # Aggregate across z-slices (median per well)
    features_list = list_features(normalized)[0]
    drop_from_meta = ['Metadata_z', 'Metadata_PlateWell', 'Metadata_plate_slice', '_col']
    meta_cols = [c for c in normalized.columns if c not in features_list and c not in drop_from_meta]
    aggregated = normalized.groupby(['Metadata_PlateWell']).agg(
        {**{c: 'first'  for c in meta_cols},
         **{c: 'median' for c in features_list}}
    ).reset_index()

    # Feature selection + clipping
    to_clip = feature_select(aggregated, features=list_features(aggregated)[0],
                             operation=['variance_threshold', 'correlation_threshold', 'drop_na_columns'])
    selected = pd.concat([
        to_clip[list_features(to_clip)[1]],
        to_clip[list_features(to_clip)[0]].clip(lower=-40, upper=40, axis=1)
    ], axis=1)

    # OutputDir comes from profiles(..., "<dir>/"), which is a Path -- and Path drops the
    # trailing slash, so string-formatting it glued the directory name onto the
    # filename and wrote the table *beside* the folder instead of inside it.
    out_path = OutputDir / f'selected_{case_name}.parquet'
    selected.to_parquet(out_path)
    print(f'  -> saved {selected.shape[0]} wells x {selected.shape[1]} cols to {out_path}')
    return selected


In [ ]:
z_all = sorted(data['Metadata_z'].unique().tolist())

print('Available z-slices:', z_all)
print('Total slices:', len(z_all))

def get_even(n, z_values):
    idx = np.linspace(0, len(z_values) - 1, n)
    idx = np.round(idx).astype(int)
    return [z_values[i] for i in idx]

# 12 planes
z_12planes = get_even(12, z_all)

# single planes (by index)
orig_planes = [2, 7, 11]
z_single = [z_all[i] for i in orig_planes if i < len(z_all)]

# sparse
z_sparse = {n: get_even(n, z_all) for n in [3, 6, 9]}

print('12 planes:', z_12planes)
print('Single planes:', z_single)
print('Sparse selections:', z_sparse)

In [ ]:
# All available planes
process_case(data, 'section1_12planes', z_slices=z_all)

In [ ]:
# Single planes — at positions equivalent to original z=2, z=7, z=11

for i in orig_planes:
    if i < len(z_all):
        z_val = z_all[i]
        process_case(data, f'section1_z{i}', z_slices=[z_val])

In [ ]:
# Sparse sampling: 3, 6, 9 slices evenly spaced across full z range
for n, z_sel in z_sparse.items():
    print(f'n={n}: {z_sel}')
    process_case(data, f'section1_sparse{n}', z_slices=z_sel)